In [ ]:
import json

import boto3
import requests


import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment-specific .env file (e.g., .env.dev, .env.test)
ENVIRONMENT = os.getenv("ENVIRONMENT", "dev")
env_file = Path(__file__).resolve().parent.parent / f".env.{ENVIRONMENT}" if "__file__" in dir() else Path(f"../.env.{ENVIRONMENT}")
load_dotenv(env_file)

# Configuration
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_URL = os.getenv("GATEWAY_API_URL", "https://your-gateway-url.com")
SECRET_ID = os.getenv("GATEWAY_SECRET_ID", "bedrock-gateway-dev-oauth-credentials")

# Extract the secret value from Secret Manager

session = boto3.Session(profile_name=os.getenv("AWS_PROFILE"), region_name=AWS_REGION)
JSON_SECRET = json.loads(session.client('secretsmanager').get_secret_value(
    SecretId=SECRET_ID)['SecretString'])
CLIENT_ID = JSON_SECRET['client_id']
CLIENT_SECRET = JSON_SECRET['client_secret']
TOKEN_URL = JSON_SECRET['token_url']

# Token Generation

In [ ]:
def generate_token():
    """Generate access token for API authentication"""

    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials",
        "audience": "bedrockproxygateway",
        "scope": "bedrockproxygateway:invoke"
    }

    response = requests.post(TOKEN_URL, data=payload)
    response.raise_for_status()
    token = response.json()['access_token']
    return f"Bearer {token}"


# Generate token
TOKEN = generate_token()
print(f"Token generated: {TOKEN[:20]}...")

# Root and Health Endpoints

## Root Endpoint

In [ ]:
response = requests.get(
    f"{API_URL}",
    headers={"Content-Type": "application/json"}
)

if response.status_code == 200:
    result = response.json()
    print(json.dumps(result, indent=2))
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## Health Endpoint

In [ ]:
response = requests.get(
    f"{API_URL}/health",
    headers={"Content-Type": "application/json"}
)

if response.status_code == 200:
    result = response.json()
    print(json.dumps(result, indent=2))
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## Redis Health Endpoint

In [ ]:
response = requests.get(
    f"{API_URL}/health/valkey",
    headers={"Content-Type": "application/json"}
)
print(response)

if response.status_code == 200:
    result = response.json()
    print(json.dumps(result, indent=2))
else:
    print(f"Error: {response.status_code}")
    print(response.text)